In [ ]:
# FTO2 NC pipeline and dashboard tables, in one run (Skywise Code Workspace). Generated by build/build_all.py from the pipeline
# notebook (06_input_data/pipeline_notebook.py): its NC selection, *_auto columns and GPS / Test merge are copied unchanged.
# What changed (20 Oct 2026):
#   - one run does both selections: the standard one, then the NCs of the GPS / Test lists it does not catch (the former
#     MISSING_IDS run); each NC keeps its origin (normal / missing_ids);
#   - the missing IDs are computed in the run from the GPS / Test lists (2026 dates), as missing_id_generator did by hand;
#   - no 1,000-row cap on the standard selection (it dropped the oldest NCs; ROW_CAP = None);
#   - the run writes the dashboard's three tables.
#
# Upload the folder notebook_kit next to this notebook. Data panel (alias = name used below):
#   read : non_conformity (as today), fto2_ai_labels (upload notebook_kit/fto2_ai_labels.csv once),
#          Work_Order_Operation as work_order_operation, MES Disruptions as mes_disruptions
#   write: fto2_dash_nc, fto2_dash_defects, fto2_dash_disruptions (sync the three to Postgres, schema foundry_sync)
#   while the current Slate app is in use (KEEP_OLD_OUTPUTS = True): fto2_dashboard_data, fto2_dashboard_data_missing_ids,
#          fto2_parsed_defects, fto2_parsed_defects_missing_ids, written as before
# The data/ folder (station mapping, GPS register, Test list) stays where it is.

In [ ]:
import sys
sys.path.insert(0, "notebook_kit")           # the dashboard's Python first: parser v4.8, CNQ model, data steps
import re
import warnings
import numpy as np
import pandas as pd
from foundry.transforms import Dataset, Column
from nc_defect_parser import explode_dataframe
import fto2_core as core

# ==========================================
# ⚙️ CONFIGURATION
# ==========================================
GPS_LIST = 'data/GPS NC INT_SERIE ELEC 2026 - NC REGISTRE.csv'   # NC number 'N°  NC / AM / AF', date ' Date SAP', 3 header rows
TEST_LIST = 'data/NC TEST - NC.csv'                              # NC number 'N° NC', date 'Dates', 2 header rows
LIST_FROM_YEAR = 2026                        # NCs of the lists dated from this year on
ROW_CAP = None                               # was 1000 (head(1000)): None keeps every NC of the standard selection
# The two other Skywise datasets (aliases in the Data panel). Empty: the model's work times, the flat alert, no disruption block.
WO_OPERATIONS = "work_order_operation"
MES_DISRUPTIONS = "mes_disruptions"
SINCE = "2025-12-01 00:00:00"                # rework operations and aléas read from this date
# True while the current Slate app reads the former outputs; False once the new dashboard has replaced it.
KEEP_OLD_OUTPUTS = True

interesting_cols = [
    # --- Basic Info & Identifiers ---
    "skywise_non_conformity_id", "notification_number", "notification_no_leading_zeros", "programme", 
    "aircraft_type", "msn", "system", "notification_type", "linked_document",
    
    # --- Status & Decision ---
    "nc_status", "system_status", "status_open_closed", "decision_status", 
    "decision_status_description_en", "decision_date", "workflow_list",

    # --- Text Descriptions ---
    "description_short", "description_long_text",
    
    # --- Defect Categorization ---
    "defect_group_description_en", "defect_code", "defect_code_description_en", 
    "ata_chapter", "cause_code", "cause_code_description_en", 
    "cause_group_description_en", "cause_description",
    
    # --- Work Location & Routing ---
    "plant", "plant_text", "work_ctr", "station", "major_component_description", 
    "constituent_assembly", "operation", "work_order", "trigger", "origin",
    
    # --- Responsibility & Liability ---
    "liability", "liability_description", "responsible_defect_level1", 
    "responsible_defect_level1_name", "user_department", "is_orgunit",
    
    # --- Parts & Quantities ---
    "defect_quantity", "priority", "returned_flag", "article_version", 
    "article_version_description",
    
    # --- Timestamps (For Lead Time Analysis) ---
    "creation_time", "changed_time", "closure_time", "completion_time"
]

In [ ]:
# =========================================================
# A. SELECTION AND *_auto COLUMNS (the former RUN_MODE is an argument; missing_ids is computed in part C)
# =========================================================
def select_ncs(RUN_MODE):
    if RUN_MODE == "STANDARD":
        # 1A. Fetch primary FTO2 plant data
        fto2_plant_df = (Dataset.get("non_conformity")
            .where(
                (Column.get("plant") == "FTO2") & 
                (Column.get("creation_time") >= "2026-01-01 00:00:00")
            )
            .select(*interesting_cols) 
            .read_table(format="pandas")
        )

        # 1B. Fetch NON-FTO2 plant data to check liability array locally
        other_plant_df = (Dataset.get("non_conformity")
            .where(
                (Column.get("plant") != "FTO2") & 
                (Column.get("creation_time") >= "2026-01-01 00:00:00")
            )
            .select(*interesting_cols)
            .read_table(format="pandas")
        )

        def check_array_for_fto2(element):
            if isinstance(element, list):
                return "FTO2" in element
            elif isinstance(element, str):
                return element == "FTO2"
            elif hasattr(element, '__iter__'): 
                return "FTO2" in list(element)
            return False

        # Isolate cross-plant rows where FTO2 is liable
        cross_plant_fto2_liable = other_plant_df[other_plant_df['liability'].apply(check_array_for_fto2)]

        # 1C. Combine both into a single working DataFrame
        non_conformity_df = pd.concat([fto2_plant_df, cross_plant_fto2_liable], ignore_index=True)

    elif RUN_MODE == "MISSING_IDS":
        print(f"Mode: MISSING_IDS - Fetching {len(missing_ids)} specific records...")

        non_conformity_df = (Dataset.get("non_conformity")
            .where(Column.get("notification_no_leading_zeros").isin(missing_ids))
            .select(*interesting_cols).read_table(format="pandas"))

    else:
        raise ValueError("Invalid RUN_MODE. Choose 'STANDARD' or 'MISSING_IDS'.")

    print(f"Total rows fetched: {len(non_conformity_df)}")
    # Safely fill nulls for string operations
    clean_cols = ['decision_status', 'workflow_list', 'article_version_description', 'user_department']
    for col in clean_cols:
        non_conformity_df[col] = non_conformity_df[col].fillna('')

    if RUN_MODE == "STANDARD":
        # Exclude 'ERMA'
        df_filtered = non_conformity_df[
            ~non_conformity_df['user_department'].str.contains('ERMA', case=False)
        ].copy()

        is_deleted = df_filtered['status_open_closed'].fillna('').str.contains('Deleted', case=False)

        #  Exclude if deleted AND plant is not FTO2
        is_not_fto2 = df_filtered['plant'] != 'FTO2'
        df_filtered = df_filtered[~(is_deleted & is_not_fto2)].copy()

        print(f"Rows moving to main processing: {len(df_filtered)}")
    else:
        df_filtered = non_conformity_df.copy()
    # --- Pre-requisite: Station Mapping ---
    station_mapping_df = pd.read_csv('data/fto2_station_mapping.csv')

    melted_list = []
    for p in ['A320', 'A330', 'A350', 'A400M']:
        temp = station_mapping_df[[f'STATION {p}', f'Désignations {p}']].dropna().copy()
        temp.columns = ['station', 'station_description']
        melted_list.append(temp)

    station_mapping_clean = pd.concat(melted_list, ignore_index=True).drop_duplicates(subset=['station'])
    df_filtered = df_filtered.merge(station_mapping_clean, on='station', how='left')
    # --- Z. Phase (Serie/REC/Hors Serie/FTI) ---
    # Default to SERIE
    df_filtered['Programme_auto'] = 'SERIE'

    # Flag REC: Catches anything with "-CRE", "-PNC", the exact word "REC", or the specific "A30-REJ-PVU2"
    rec_pattern = r'\bREC\b|-CRE|-PNC|A30-REJ-PVU2'
    is_rec = df_filtered['workflow_list'].fillna('').str.contains(rec_pattern, case=False, regex=True)
    df_filtered.loc[is_rec, 'Programme_auto'] = 'REC'

    # Flag FTI: Catches any assembly starting with 'WY' or 'WA' (e.g., WA999)
    is_fti = df_filtered['constituent_assembly'].fillna('').str.upper().str.startswith(('WY', 'WA'))
    df_filtered.loc[is_fti, 'Programme_auto'] = 'FTI'

    # --- A. Statut Décision ---
    statut_map = {
        'RWK': 'Rework', 
        'REJ': 'Reject', 
        'EXC': 'Echange', 
        'NDC': 'DQN', 
        'SCR': 'Scrap',
        'ACC': 'Accept as is i.a.w design criteria',
        'PNC': 'Potential Non Conformity',
        'DQN': 'Request for Design Change'
    }
    df_filtered['Statut_decision_auto'] = df_filtered['decision_status'].map(statut_map).fillna('Unknown')

    # --- B. Origin ---
    # Rule: OLMEP14/AIRBUS OPERATIONS SAS = Test, everything else = Quality
    df_filtered['Origin_auto'] = np.where(
        df_filtered['user_department'].str.strip().str.upper() == 'OLMEP14/AIRBUS OPERATIONS SAS', 
        'Test', 
        'Qualite'
    )

    # --- 3. Type NC (UPDATED WITH PRIORITIES) ---
    is_rew_ligne = df_filtered['workflow_list'].str.startswith('REW-LIGNE')
    is_test = df_filtered['workflow_list'].str.contains('TEST', case=False) 
    is_derichebourg = df_filtered['user_department'].str.contains('DERICHEBOURG', case=False)

    desc_short = df_filtered['description_short'].fillna('')
    desc_article = df_filtered['article_version_description'].fillna('')
    wf_list = df_filtered['workflow_list'].fillna('')

    # -- Existing Cables Logic --
    is_ca_workflow = wf_list.str.contains(r'^R[E]?W-CA-IMP', flags=re.IGNORECASE)
    is_label_plate = desc_short.str.contains(r'etiquette|plaquette', flags=re.IGNORECASE) | \
                     desc_article.str.contains(r'etiquette|plaquette', flags=re.IGNORECASE)
    is_explicit_cable = desc_short.str.contains(r'c[aâ]ble|harnais|wire', flags=re.IGNORECASE) | \
                        desc_article.str.contains(r'c[aâ]ble|harnais|wire', flags=re.IGNORECASE)
    is_misrouted_cable = is_explicit_cable & wf_list.str.startswith('REW-MP-IMP')
    cond_cables_accurate = (is_ca_workflow & ~is_label_plate) | is_misrouted_cable

    # -- NEW HIGH-PRIORITY RULES --
    cond_deleted_fto2 = df_filtered['status_open_closed'].fillna('').str.contains('Deleted', case=False) & (df_filtered['plant'] == 'FTO2')
    cond_test_new = df_filtered['Origin_auto'] == 'Test'
    cond_retour_client = ~df_filtered['plant'].fillna('').isin(['FTO2', 'FT02'])
    cond_fournisseur_new = (df_filtered['liability_description'].fillna('') == '[SUPPLIER]') | \
                           (df_filtered['responsible_defect_level1'].fillna('') == 'Y4')

    # Apply all rules in exact priority order
    conds_type = [
        cond_deleted_fto2,                                          # �� 0. Deleted FTO2 -> QUALITE
        cond_test_new,                                              # 1. TEST
        cond_retour_client,                                         # 2. Retour client
        cond_fournisseur_new,                                       # 3. Fournisseur
        cond_cables_accurate,                                       # 4. Cables
        wf_list.str.startswith('REJ-LO-BP') | (is_rew_ligne & is_derichebourg), # 5. Interne
        wf_list.str.startswith('LIGNE-F-DQN'),                      # 6. Demande de DQN
        is_rew_ligne & ~is_derichebourg,                            # 7. Controle Final
        is_test                                                     # 8. Controle Final
    ]

    choices_type = [
        'QUALITE',          # �� Added for cond_deleted_fto2
        'TEST', 
        'NC Retour client', 
        'NC Fournisseur', 
        'CABLES', 
        'NC Interne', 
        'DEMANDE DE DQN', 
        'CONTROLE FINAL', 
        'CONTROLE FINAL'
    ]

    # Set default to 'NC Interne'
    df_filtered['Type_NC_auto'] = np.select(conds_type, choices_type, default='NC Interne')

    # --- D. Produit (With Dynamic VB/VU Regex & Expanded Rules) ---
    # Combine both columns so the regex searches the article first, then the station as a fallback
    desc = df_filtered['article_version_description'].fillna('') + " | " + df_filtered['station_description'].fillna('')
    prog = df_filtered['programme'].fillna('')

    is_a400 = prog.isin(['MT', 'A400M'])
    is_mip_desc = desc.str.contains('MIP|1101VB|1212VB|2200VB', case=False, regex=True)
    vb_vu_extracted = desc.str.extract(r'(\d{3,4}V[UB])', expand=False, flags=re.IGNORECASE)

    conds_prod = [
        desc.str.contains('4714VB|4715VB|4716VB|4717VB|E-RUDDER', case=False, regex=True),
        desc.str.contains('5883VB|5893VB|MDCD', case=False, regex=True),
        desc.str.contains('OHP', case=False, regex=False),
        desc.str.contains('30[0-5]VU|M8753254100000', case=False, regex=True),
        desc.str.contains('WIRE BUNDLE|MFRL', case=False, regex=True),
        desc.str.contains('ELEC ASSY', case=False, regex=False),
        is_mip_desc & is_a400,
        is_mip_desc & ~is_a400,
        vb_vu_extracted.notna() 
    ]

    choices_prod = [
        'E-RUDDER', 'MDCD', 'OHP', '300VU', 'MFRL', 'PTVU', 
        'MIP A400M', 'MIP A350', vb_vu_extracted 
    ]

    df_filtered['Produit_auto'] = np.select(conds_prod, choices_prod, default='Other')

    # Sort; the former 1,000-row cap is ROW_CAP (None: every NC of the selection)
    final_df = df_filtered.sort_values(by='creation_time', ascending=False)
    if ROW_CAP:
        final_df = final_df.head(ROW_CAP)
    return final_df

In [ ]:
# =========================================================
# 1. HELPER FUNCTION: Load & Clean IDs safely
# =========================================================
def load_external_nc_csv(filepath, skiprows, id_col):
    """Loads CSV, safely extracts the ID, drops nulls, and forces a clean string."""
    try:
        df = pd.read_csv(filepath, skiprows=skiprows, low_memory=False)
        df = df.dropna(subset=[id_col]).copy()
        
        # Clean the ID: convert to string, strip '.0' if pandas inferred a float
        df['notification_number'] = df[id_col].astype(str).str.strip().str.replace(r'\.0$', '', regex=True)
        
        # Keep only valid numeric strings, drop duplicates
        df = df[df['notification_number'].str.isnumeric()]
        df = df.drop_duplicates(subset=['notification_number'])
        return df
    except Exception as e:
        print(f"❌ Error loading {filepath}: {e}")
        return pd.DataFrame()

# =========================================================
# 2. LOAD AND NORMALIZE DATASETS
# =========================================================

# --- A. Test Data ---
test_df = load_external_nc_csv('data/NC TEST - NC.csv', skiprows=2, id_col='N° NC')

if not test_df.empty:
    # Merge the two Test product columns into one string for human readability
    test_df['Ext_Produit'] = test_df['Produit'].fillna('') + " " + test_df['VU\nuniquement PTVU'].fillna('')
    test_df['Ext_Produit'] = test_df['Ext_Produit'].str.strip()
    
    # Rename shared columns to standard "Ext_" prefix
    test_df = test_df.rename(columns={
        'Nb défaut': 'Ext_Nb_Defauts',
        'Défaut majeur': 'Ext_Defaut_Constate'
    })
    test_df['Ext_Source'] = 'Test'
    
    # Isolate only the columns we need
    test_clean = test_df[['notification_number', 'Ext_Produit', 'Ext_Nb_Defauts', 'Ext_Defaut_Constate', 'Ext_Source']].copy()
else:
    test_clean = pd.DataFrame(columns=['notification_number', 'Ext_Produit', 'Ext_Nb_Defauts', 'Ext_Defaut_Constate', 'Ext_Source'])

# --- B. GPS Data ---
gps_df = load_external_nc_csv('data/GPS NC INT_SERIE ELEC 2026 - NC REGISTRE.csv', skiprows=3, id_col='N°  NC / AM / AF')

if not gps_df.empty:
    # Rename shared columns to standard "Ext_" prefix
    gps_df = gps_df.rename(columns={
        'Produit': 'Ext_Produit',
        'Nombre total retouches': 'Ext_Nb_Defauts',
        'Non conformité constatée': 'Ext_Defaut_Constate',
        
        # Keep GPS-specific columns clearly labeled
        'Type NC': 'GPS_Type_NC',
        'Statut décision': 'GPS_Statut_decision',
        'IMPUTATION \n(après analyse)': 'GPS_Imputation',
        'Nombre de retouches majeures': 'GPS_Retouches_majeures',
        'Nombre de retouches mineures': 'GPS_Retouches_mineures'
    })
    gps_df['Ext_Source'] = 'GPS'
    
    gps_cols_to_keep = ['notification_number', 'Ext_Produit', 'Ext_Nb_Defauts', 'Ext_Defaut_Constate', 'Ext_Source', 
                        'GPS_Type_NC', 'GPS_Statut_decision', 'GPS_Imputation', 'GPS_Retouches_majeures', 'GPS_Retouches_mineures']
    gps_clean = gps_df[gps_cols_to_keep].copy()
else:
    gps_clean = pd.DataFrame()

# =========================================================
# 3. MERGE & CONSOLIDATE SHARED COLUMNS
# =========================================================

# Outer join keeps everything from both files
external_combined = pd.merge(
    test_clean, 
    gps_clean, 
    on='notification_number', 
    how='outer', 
    suffixes=('_test', '_gps')
)

# Function to safely combine overlapping columns, prioritizing Test, then GPS
def coalesce_cols(df, base_col_name):
    col_test = f"{base_col_name}_test"
    col_gps = f"{base_col_name}_gps"
    
    if col_test in df.columns and col_gps in df.columns:
        # combine_first fills NaNs in the Test column with values from the GPS column
        df[base_col_name] = df[col_test].combine_first(df[col_gps])
        df.drop(columns=[col_test, col_gps], inplace=True)
    elif col_test in df.columns:
        df.rename(columns={col_test: base_col_name}, inplace=True)
    elif col_gps in df.columns:
        df.rename(columns={col_gps: base_col_name}, inplace=True)

# Apply consolidation
coalesce_cols(external_combined, 'Ext_Produit')
coalesce_cols(external_combined, 'Ext_Nb_Defauts')
coalesce_cols(external_combined, 'Ext_Defaut_Constate')

# Resolve Source logic (Identify if it somehow ended up in both lists)
def resolve_source(row):
    src_test = row.get('Ext_Source_test')
    src_gps = row.get('Ext_Source_gps')
    
    if pd.notna(src_test) and pd.notna(src_gps):
        warnings.warn(f"DATA ERROR: NC {row['notification_number']} found in BOTH Test and GPS files.")
        return 'Both (Error)'
    elif pd.notna(src_test): return src_test
    elif pd.notna(src_gps): return src_gps
    return 'Unknown External'

external_combined['Ext_Source'] = external_combined.apply(resolve_source, axis=1)
if 'Ext_Source_test' in external_combined.columns:
    external_combined = external_combined.drop(columns=['Ext_Source_test', 'Ext_Source_gps'])



def add_external(final_df):
    # =========================================================
    # 4. FINAL MERGE WITH MAIN PIPELINE (final_df)
    # =========================================================

    # Force strict string typing to ensure a flawless join
    external_combined['notification_number'] = external_combined['notification_number'].astype(str)
    final_df['notification_number'] = final_df['notification_number'].astype('Int64').astype(str)

    final_df = final_df.merge(external_combined, on='notification_number', how='left')

    # Fill empty Sources for Skywise records that weren't in the CSVs
    final_df['Ext_Source'] = final_df['Ext_Source'].fillna('Not in external lists')

    # Force all added external columns to string to prevent Foundry schema errors
    external_str_cols = [
        'Ext_Produit', 'Ext_Nb_Defauts', 'Ext_Defaut_Constate', 'Ext_Source',
        'GPS_Type_NC', 'GPS_Statut_decision', 'GPS_Imputation', 
        'GPS_Retouches_majeures', 'GPS_Retouches_mineures'
    ]

    for col in external_str_cols:
        if col in final_df.columns:
            final_df[col] = final_df[col].fillna('').astype(str)

    # =========================================================
    # FORCE ALIGNMENT FOR SQL UNION
    # =========================================================
    if 'is_orgunit' in final_df.columns:
        # Force to String (matches fto2_plant)
        final_df['is_orgunit'] = final_df['is_orgunit'].astype(str)

    if 'defect_quantity' in final_df.columns:
        # Force to Float/Double (matches fto2_plant)
        final_df['defect_quantity'] = final_df['defect_quantity'].astype(float)
    return final_df

In [ ]:
# =========================================================
# B. THE NC NUMBERS OF THE GPS / TEST LISTS (from missing_id_generator)
# =========================================================
def extract_nc_ids(file_name, id_column, date_column, skiprows=0):
    """NC numbers of a list whose date is in LIST_FROM_YEAR or later (dates dd/mm/yyyy)."""
    try:
        df = pd.read_csv(file_name, skiprows=skiprows)
        if id_column not in df.columns or date_column not in df.columns:
            print(f"❌ Error: Missing columns in {file_name}.")
            return set()
        dates = pd.to_datetime(df[date_column], format='%d/%m/%Y', errors='coerce')
        ids = df.loc[dates.dt.year >= LIST_FROM_YEAR, id_column].dropna().astype(str).str.strip().str.replace(r'\.0$', '', regex=True)
        return set(ids[ids.str.isdigit()])
    except Exception as e:
        print(f"❌ Error processing {file_name}: {e}")
        return set()

list_ids = extract_nc_ids(GPS_LIST, 'N°  NC / AM / AF', ' Date SAP', skiprows=3) | extract_nc_ids(TEST_LIST, 'N° NC', 'Dates', skiprows=2)

In [ ]:
# =========================================================
# C. BOTH SELECTIONS IN ONE RUN; EACH NC KEEPS ITS ORIGIN
# =========================================================
standard_df = add_external(select_ncs("STANDARD"))
in_standard = set(standard_df['notification_number'].astype(str).str.lstrip('0'))
missing_ids = sorted(i.lstrip('0') for i in list_ids if i.lstrip('0') not in in_standard)   # on the lists, not caught by the selection
print(f"GPS / Test lists: {len(list_ids)} NCs from {LIST_FROM_YEAR}; {len(missing_ids)} not in the standard selection, fetched by number")
runs = [("fto2_dashboard_data", standard_df)]
if missing_ids:
    missing_df = add_external(select_ncs("MISSING_IDS"))
    found = set(missing_df['notification_number'].astype(str).str.lstrip('0'))
    print(f"   found in non_conformity: {len(found)}; not in Skywise at all: {len(set(missing_ids) - found)}")
    runs.append(("fto2_dashboard_data_missing_ids", missing_df))

frames = []
for name, final_df in runs:
    final_df['Ext_Nb_Defauts'] = pd.to_numeric(final_df['Ext_Nb_Defauts'], errors='coerce')
    if KEEP_OLD_OUTPUTS:   # the former outputs, as before, for the current Slate app
        enriched_nc_df, defects_df = explode_dataframe(df=final_df, text_col="description_long_text", id_col="notification_number", manual_count_col="Ext_Nb_Defauts")
        for col in ['code', 'fin', 'text', 'source_format', 'mode_erreur', 'type_reporting', 'loc', 'severite', 'sub_code', 'confidence', 'resolved_by', 'sap_type', 'severite_off']:
            if col in defects_df.columns:
                defects_df[col] = defects_df[col].astype(str).fillna('')
        Dataset.get(name.replace("fto2_dashboard_data", "fto2_parsed_defects")).write_table(defects_df)
        Dataset.get(name).write_table(final_df)
        print(f"✅ {name}: {len(final_df)} NCs, {len(defects_df)} defects (former outputs)")
    final_df = final_df.copy()
    final_df["_source"] = name               # origin of each NC: normal, or missing_ids when the name holds "missing"
    frames.append(final_df)

In [ ]:
# =========================================================
# D. THE DASHBOARD'S THREE TABLES
# =========================================================
labels = Dataset.get("fto2_ai_labels").read_table(format="pandas")
ops = dis = None
if WO_OPERATIONS:
    ops = (Dataset.get(WO_OPERATIONS)
           .where((Column.get("plant") == "FTO2") & (Column.get("order_type") == "S03"))
           .select("order_number", "order_type", "operation_number", "operation_short_text", "plant", "labor_standard_value",
                   "unit_of_measure_for_the_standard_value_labor", "setup_standard_value", "unit_of_measure_for_the_standard_value_setup")
           .read_table(format="pandas"))
if MES_DISRUPTIONS:
    dis = (Dataset.get(MES_DISRUPTIONS)
           .where((Column.get("plant") == "FTO2") & (Column.get("open_date_time") >= SINCE))
           .select("disruption_id", "plant", "physical_station", "work_order", "wo_designation", "ds_material", "type_desc", "open_date_time",
                   "open_date_time_tz", "status", "time_lost", "program", "comment")
           .read_table(format="pandas"))
nc_t, def_t, dis_t, info, to_label = core.build(frames, labels, ref_dir="notebook_kit/ref", ops=ops, dis=dis)

Dataset.get("fto2_dash_nc").write_table(nc_t)
Dataset.get("fto2_dash_defects").write_table(def_t)
Dataset.get("fto2_dash_disruptions").write_table(dis_t)
print("✅ FTO2 dashboard tables:", info["nc_table"], "NCs,", info["defect_table"], "defect lines,", info["disruption_table"], "aléas; level-3 CNQ on AEC:", info["level3_aec"], "EUR")
print("NCs created", info["created_from"], "to", info["created_to"], "| origin:", info["origin"], "| defect lines without an AI label (parser label used):", info["to_label"])
print("rework orders:", info["rework_orders"], "| aléas:", info.get("aleas", "not read"))

# lines for the next AI pass (once IT has approved the tool): a CSV to download from the workspace. Each line comes with its
# full text (snippet) and the NC's long text; the labels go back into fto2_ai_labels with the same notification_number, seq, snippet.
to_label.to_csv("fto2_lines_to_label.csv", index=False)